In [1]:
import os
import pandas as pd
import numpy as np

CFG = {
    "weights_spec": "",  # not used in the baseline but kept for compatibility
    "weights_eeg": "",
    "weights_mix": "",
}


def _list_weights(path):
    return (
        [os.path.join(path, x) for x in sorted(os.listdir(path))]
        if os.path.isdir(path)
        else []
    )


CFG["weights_spec"] = _list_weights(CFG["weights_spec"])
CFG["weights_eeg"] = _list_weights(CFG["weights_eeg"])
CFG["weights_mix"] = _list_weights(CFG["weights_mix"])

CFG




{'weights_spec': [], 'weights_eeg': [], 'weights_mix': []}

In [2]:
TRAIN_PATH = "/kaggle/input/hms-harmful-brain-activity-classification/train.csv"
TEST_PATH = "/kaggle/input/hms-harmful-brain-activity-classification/test.csv"
SUBMISSION_PATH = "submission.csv"

TARGETS = [
    "seizure_vote",
    "lpd_vote",
    "gpd_vote",
    "lrda_vote",
    "grda_vote",
    "other_vote",
]

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

# Global class prior (Laplace smoothed)
total_votes = train_df[TARGETS].astype(np.float64).sum()
class_prior = (total_votes + 1) / (total_votes.sum() + len(TARGETS))  # shape (6,)

# Stronger Laplace smoothing for patient‑specific posteriors
gamma = 100.0  # increased from 10 to pull patient posteriors toward the global prior

patient_posterior = {}
patient_counts = {}

for pid, group in train_df.groupby("patient_id"):
    votes = group[TARGETS].astype(np.float64).sum()  # raw vote sums per class
    vote_sum = votes.sum()
    if vote_sum == 0:
        posterior = class_prior.copy()
    else:
        posterior = (votes + gamma * class_prior) / (vote_sum + gamma)
    patient_posterior[pid] = posterior
    patient_counts[pid] = len(group)

max_count = max(patient_counts.values()) if patient_counts else 0
log_max = np.log1p(max_count)

epsilon = 1e-6
pred_list = []

for _, row in test_df.iterrows():
    pid = row["patient_id"]
    posterior = patient_posterior.get(pid, class_prior)

    count = patient_counts.get(pid, 0)
    # Reduce patient influence: halve the log‑scaled weight and cap at 0.5
    weight = (np.log1p(count) / (2 * log_max)) if log_max > 0 else 0.0
    weight = min(weight, 0.5)

    pred = weight * posterior + (1 - weight) * class_prior

    pred = np.clip(pred, epsilon, None)
    pred = pred**0.9  # stronger flattening than original 0.95
    pred = pred / pred.sum()
    pred_list.append(pred)

predictions = np.vstack(pred_list).astype(np.float32)

submission_df = pd.DataFrame(predictions, columns=TARGETS)
submission_df.insert(0, "eeg_id", test_df["eeg_id"])
submission_df.to_csv(SUBMISSION_PATH, index=False)

print(f"Submission written to {SUBMISSION_PATH}")

Submission written to submission.csv
